# P108 — CAP doce años después: cómo han cambiado las «reglas»

## 1. Título y paper

**Paper:** *CAP Twelve Years Later: How the «Rules» Have Changed*  
**Autoría:** Eric Brewer  
**Año y venue:** 2012 · IEEE Computer, 45(2), 23–29  
**Nivel:** L2 · **Motor:** `resiliencia`  
**Ficha completa:** [`P108_cap`](../../papers/foundational/P108_cap/README.md)

**Hito:** Corrige la lectura simplista de su propio teorema: no se eligen dos de tres, se elige por operación y solo mientras dura la partición.

- [doi:10.1109/MC.2012.37](https://doi.org/10.1109/MC.2012.37)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El teorema CAP se citaba como «elige dos de consistencia, disponibilidad y tolerancia a particiones», y eso llevó a decisiones de arquitectura globales y rígidas: sistemas enteros declarados AP o CP.
2. Ejecutar una implementación mínima de la propuesta: Reformularlo con precisión: la tolerancia a particiones no es opcional, y la elección entre consistencia y disponibilidad solo aplica **durante** una partición. Se decide por operación, y hay que diseñar explícitamente la detección de la partición, el modo degradado y la reconciliación posterior.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Brewer (2000), la conjetura original
- Gilbert y Lynch (2002), la demostración formal


## 4. Intuición

El teorema CAP se cita como «elige dos de tres» y esa lectura ha producido decisiones de arquitectura equivocadas durante una década. Brewer corrige su propia formulación doce años después: la tolerancia a particiones no es opcional, y la elección solo existe **mientras dura la partición**.


## 5. Concepto mínimo

```text
Sin partición  →  no hay que elegir nada: consistencia Y disponibilidad
Con partición  →  hay que elegir, y se elige POR OPERACIÓN

    CP: rechazar escrituras para no divergir
    AP: aceptarlas y reconciliar después

Diseñar explícitamente: detección · modo degradado · reconciliación
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('resiliencia', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué pasa con las réplicas bajo la estrategia CP?
2. ¿Y bajo AP?
3. ¿Qué pasa si se reintenta una operación sin clave de idempotencia?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('resiliencia', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('resiliencia', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con CP se rechazan **4 escrituras** durante la partición y las réplicas nunca divergen. Con AP se sirven **4 escrituras** y las réplicas llegan a divergir en **2**. Y reintentar un cobro sin clave de idempotencia cobra **3 veces** en lugar de una.


## 10. Comentario pedagógico

Ninguna de las dos estrategias es la correcta en abstracto: cobrar una tarjeta pide CP y mostrar un contador de «me gusta» pide AP, **en el mismo sistema**. El matiz de 2012 es que la elección no es una etiqueta de la arquitectura sino una decisión por operación, y que hay que diseñar explícitamente qué se hace durante la partición y cómo se reconcilia después.


## 11. Error o anti-patrón deliberado

Anti-patrón: reintentar sin clave de idempotencia.


In [ ]:
print('Un reintento tras un timeout no sabe si la operacion se ejecuto o no.')
print('Sin clave de idempotencia, reintentar tres veces cobra tres veces.')
print('La resiliencia no es reintentar: es que repetir NO cambie el resultado.')

## 12. Corrección

El diseño que hace seguro el reintento:


In [ ]:
r = run_paper_lab('resiliencia', seed=7)['result']
print('CP:', {k: v for k, v in r['estrategia_CP'].items() if k != 'historia'})
print('AP:', {k: v for k, v in r['estrategia_AP'].items() if k != 'historia'})
print('idempotencia:', r['idempotencia'])

## 13. Desafío guiado

Elige tres operaciones de un sistema que conozcas y decide, para cada una, si durante una partición conviene CP o AP. Justifica cada decisión.


In [ ]:
r = run_paper_lab('resiliencia', seed=3)['result']
show(r)

## 14. Desafío autónomo

Revisa un servicio tuyo: ¿qué pasa si se parte la red entre él y su base de datos? Documenta el modo degradado, el criterio de reconciliación y si sus operaciones son idempotentes.


## 15. Evidencia de aprendizaje

Guarda la comparación entre CP y AP con la divergencia máxima, y tu tabla de decisión por operación.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P108_cap/README.md) · evaluación formal: [`assessments/papers/P108_cap.md`](../../assessments/papers/P108_cap.md)


## 16. Cierre

El sistema ya sobrevive a la partición. Queda el problema de rendimiento que aparece cuando una petición depende de muchos servidores a la vez.


## 17. Conexión con el siguiente hito

- P107
- P109

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
